---
title: "5. Images, video and AI"
subtitle: "Diffusion from still pictures to moving ones"
description: "How diffusion models turn noise into pictures, the control signals that make them steerable, how to hold a series together, what changes when the picture has to move, and whether an image model copies what it was trained on."
---

Your phone already runs a diffusion model. The one-tap object removal in its photo editor, the button that extends a photograph past its edges, and the clip that animates a still into a few seconds of motion are all the same machinery, hidden behind a thumb-sized control. This chapter opens that control up. It works at the model and interface layers of [the five layers](intro.ipynb#five-layers). An image tool exposes a seed, a guidance number, a step count, and a slot for a reference picture, and none of those make sense until you know what the model is doing while it runs.

Pictures are also where Creative AI announced itself loudest. In 2022 three text-to-image systems reached the public within a few months of each other, and they changed picture-making faster than any tool since the smartphone camera. The first credible [text-to-video](https://en.wikipedia.org/wiki/Text-to-video_model) systems came two years later, and the delay is informative: moving pictures are the same problem with a time axis added, and the time axis is where the machinery, not the interface, sets the limits.

The aim is narrower than learning to prompt an image tool. It is an accurate account of what happens between the prompt and the picture, and then between one frame and the next. When an output is wrong, you can say *which knob* is wrong. You can make four images that belong together rather than four that merely came from the same sentence, and you can plan three shots that cut together rather than three clips that each looked good alone.

```{admonition} Question
:class: question
Before you read on, write down in one sentence where you think the picture comes from. Does the model hold a library of images somewhere and assemble from it, or does it hold something else? Keep your answer, and hold it against the critical look at the end of this chapter.
```

## How diffusion works (in pictures)

The image models that reached the public in 2022, and nearly everything built on them since, are [diffusion models](https://en.wikipedia.org/wiki/Diffusion_model) [@Ho2020Diffusion; @Rombach2022LatentDiffusion]. The idea is simpler than it looks.

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

rng = np.random.default_rng(3)

# A small synthetic "photograph" to push noise into and back out of.
n = 64
yy, xx = np.mgrid[0:n, 0:n] / (n - 1)
red = np.exp(-(((xx - 0.35) ** 2 + (yy - 0.55) ** 2) / 0.05))
green = np.exp(-(((xx - 0.68) ** 2 + (yy - 0.40) ** 2) / 0.04))
image = np.clip(np.stack([0.15 + 0.8 * red, 0.25 + 0.7 * green, 0.35 + 0.5 * yy], -1), 0, 1)
noise = rng.random((n, n, 3))

levels = [0.0, 0.15, 0.35, 0.6, 0.85, 1.0]
frames = [np.clip(np.sqrt(1 - a) * image + np.sqrt(a) * noise, 0, 1) for a in levels]
under = ["image", "+ a bit", "+ more", "noisier", "noisier", "pure noise"]

fig, ax = plt.subplots(figsize=(10, 5))
width, gap = 1.6, 1.95
navy, rose = "#1f2545", "#e8556d"

for i, frame in enumerate(frames):
    x = 0.3 + i * gap
    ax.imshow(frame, extent=[x, x + width, 3.5, 5.1], aspect="auto", zorder=2)
    ax.imshow(frame, extent=[x, x + width, 0.9, 1.9], aspect="auto", zorder=2)
    ax.text(x + width / 2, 3.3, under[i], ha="center", va="top", fontsize=10, color=navy)
    if i < len(frames) - 1:
        ax.annotate("", xy=(x + gap, 4.3), xytext=(x + width, 4.3),
                    arrowprops=dict(arrowstyle="-|>", color=navy, lw=1.8))
        ax.annotate("", xy=(x + width, 1.4), xytext=(x + gap, 1.4),
                    arrowprops=dict(arrowstyle="-|>", color=rose, lw=1.8))

ax.text(0.3, 5.6, "Forward (training): add noise step by step",
        fontsize=11, fontweight="bold", color=navy)
ax.text(0.3, 2.4, "Reverse (inference): denoise from random noise",
        fontsize=11, fontweight="bold", color=rose)
ax.text(1.1, 0.7, "image", ha="center", va="top", fontsize=10, color=navy)
ax.text(11.65 - width / 2, 0.7, "noise", ha="center", va="top", fontsize=10, color=navy)

ax.set_xlim(0, 12)
ax.set_ylim(0.1, 6.0)
ax.axis("off")
plt.tight_layout()
plt.show()

*Figure: The top row is forward diffusion, which starts from a real image and adds noise step by step until nothing is left. The bottom row is reverse diffusion, which starts from pure noise and lets a neural network remove it step by step, conditioned on a text prompt.*

The training procedure has two halves.

1. **Forward.** Take a real image. Add a tiny bit of noise. Add a tiny bit more. Repeat many times until the image is pure static. Nothing is learned in this half; it is a recipe for making training pairs.
2. **Reverse.** Train a neural network to *undo* one step of noise at a time. Given a noisy image and a number saying how noisy it is, predict what was added.

Once the network is trained you can run the reverse half *from scratch*. Start with pure noise, denoise step by step, and a coherent image emerges out of the static. Nothing was retrieved. The picture is the product of a network that has learned, across many millions of examples, what a slightly less noisy version of any given mess usually looks like.

On its own that gives you a random plausible picture, which is a demonstration rather than a tool. What makes it usable is conditioning, introduced in chapter [2](how-it-works.ipynb): an extra input fed to the model alongside the noise in order to bias what comes out. For images the most common one is a text prompt, turned into numbers by a text encoder trained to place pictures and their captions near each other in the same space [@Radford2021CLIP], and injected at every denoising step.

Four practical consequences follow from that loop, and between them they cover most of what an image tool offers.

- The model can be reused for image-to-image by starting from a *partially noisy* version of an existing picture rather than from pure noise. Less added noise means more faithfulness to the input.
- The model can be reused for [inpainting](https://en.wikipedia.org/wiki/Inpainting) by denoising only a masked region and leaving the rest of the pixels alone.
- The model can be reused for outpainting by treating an extension of the canvas as a masked region to be filled.
- The number of steps trades quality against time. Many models released since 2023 produce usable results in four to eight steps using distilled samplers, where the first generation needed fifty.

Latent diffusion [@Rombach2022LatentDiffusion] added the efficiency trick that put these models on ordinary hardware. The denoising does not happen in pixel space, where a single image is millions of numbers, but in a compressed [latent space](https://en.wikipedia.org/wiki/Latent_space) of thousands of numbers. A separate small network encodes into that space and decodes out of it. The picture is only rendered at full resolution at the very end. This is the difference between a model that needs a data centre and a model that runs on a laptop, and it is the main reason open-weight image models exist at all.

In [ ]:
from matplotlib.patches import Circle, FancyBboxPatch, Polygon, Rectangle

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"

fig, ax = plt.subplots(figsize=(10, 4.4))
ax.set_aspect("equal")


def picture(x, y, size, edge):
    """A picture glyph: a frame holding a sun and two hills."""
    ax.add_patch(Rectangle((x, y), size, size, facecolor="white", edgecolor=edge, linewidth=1.8))
    ax.add_patch(Circle((x + 0.28 * size, y + 0.72 * size), 0.09 * size,
                        facecolor="none", edgecolor=edge, linewidth=1.6))
    ax.add_patch(Polygon([(x + 0.06 * size, y + 0.1 * size), (x + 0.38 * size, y + 0.55 * size),
                          (x + 0.65 * size, y + 0.1 * size)], closed=True,
                         facecolor="none", edgecolor=edge, linewidth=1.6))
    ax.add_patch(Polygon([(x + 0.5 * size, y + 0.1 * size), (x + 0.73 * size, y + 0.42 * size),
                          (x + 0.95 * size, y + 0.1 * size)], closed=True,
                         facecolor="none", edgecolor=edge, linewidth=1.6))


def box(x, y, w, h, text):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.12",
                                facecolor="#f4f1fa", edgecolor=purple, linewidth=1.8))
    ax.text(x + w / 2, y + h / 2, text, ha="center", va="center", fontsize=11, color=navy)


def arrow(x1, x2, y, colour=navy, style="-|>", dashed=False):
    ax.annotate("", xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle=style, color=colour, lw=1.8,
                                linestyle="--" if dashed else "-"))


picture(0.4, 1.5, 1.7, purple)
ax.text(1.25, 1.25, "millions of numbers", ha="center", va="top", fontsize=10, color=navy)
arrow(2.1, 2.6, 2.35)
box(2.6, 2.0, 1.6, 0.7, "encoder")
arrow(4.2, 4.9, 2.35)

for k in range(3):
    ax.add_patch(Rectangle((4.9 + k * 0.85, 2.0), 0.7, 0.7,
                           facecolor="white", edgecolor=purple, linewidth=1.8))
ax.text(6.0, 2.9, "thousands of numbers", ha="center", va="bottom", fontsize=10, color=navy)
ax.plot([7.3, 7.3, 4.9, 4.9], [2.0, 1.55, 1.55, 2.0], color=rose, lw=1.8, ls="--")
ax.annotate("", xy=(4.9, 2.0), xytext=(4.9, 1.55),
            arrowprops=dict(arrowstyle="-|>", color=rose, lw=1.8))
ax.text(6.1, 1.3, "denoise, step by step, under the prompt",
        ha="center", va="top", fontsize=10, color=rose)

arrow(7.5, 8.0, 2.35)
box(8.0, 2.0, 1.6, 0.7, "decoder")
arrow(9.6, 10.1, 2.35)
picture(10.1, 1.5, 1.7, rose)
ax.text(10.95, 1.25, "rendered once, at the end", ha="center", va="top", fontsize=10, color=navy)

ax.text(1.25, 3.5, "Pixel space", ha="center", fontsize=11, fontweight="bold", color=purple)
ax.text(6.0, 3.5, "Latent space", ha="center", fontsize=11, fontweight="bold", color=purple)
ax.text(10.95, 3.5, "Pixel space", ha="center", fontsize=11, fontweight="bold", color=purple)
ax.text(6.1, 0.35, "The expensive loop runs on the small representation, "
        "which is why these models fit on a laptop.",
        ha="center", fontsize=10, color=navy)

ax.set_xlim(0, 12.2)
ax.set_ylim(0.1, 3.9)
ax.axis("off")
plt.tight_layout()
plt.show()

*Figure: An encoder compresses the picture into a small latent representation, the denoising loop runs there, and a decoder renders the picture at full resolution only at the end.*

The most recent generation of image models uses flow matching or rectified flows rather than plain diffusion [@Esser2024SD3]. The mathematics is tidier and the sampling is shorter, but the intuition in the figure above still holds: something structureless is pushed, step by step, towards something structured, under the influence of your prompt.

## The vocabulary of text-to-image

Every image tool exposes roughly the same small set of knobs, under slightly different names. Learning them once transfers everywhere.

- **Prompt.** What you want.
- **Negative prompt.** What you do not want. Often more powerful than people expect, and not supported by every tool.
- **Aspect ratio.** Square, landscape, portrait. Some compositions work in one and collapse in another, so this is a compositional decision rather than an export setting.
- **Guidance, sometimes labelled CFG scale, short for classifier-free guidance.** How strictly the model should obey the prompt. Typical defaults sit around five to nine. Higher gives you a more literal reading of your words and often a harsh, oversaturated picture; lower gives you a looser, more varied, sometimes more interesting one.
- **Steps.** The number of denoising steps. A few dozen for a full-quality model and a handful for a distilled one.
- **Seed.** The [random number](https://en.wikipedia.org/wiki/Random_seed) that generates the starting noise. The same prompt with the same seed in the same model gives you the same image every time, which is what makes controlled comparison possible at all.
- **Sampler or scheduler.** The algorithm that walks the denoising path. It affects both style and how quickly the image settles.
- **Reference image.** An extra conditioning input, covered below.

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"

guidance = np.linspace(1, 20, 300)
adherence = 1 - np.exp(-(guidance - 1) / 3.0)
naturalness = 1 / (1 + np.exp((guidance - 9.5) / 2.0))

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.axvspan(5, 10, color="#f4f1fa", zorder=0)
ax.plot(guidance, adherence, color=purple, lw=2.6, label="does it follow the prompt")
ax.plot(guidance, naturalness, color=rose, lw=2.6, label="does it still look natural")

ax.text(7.5, 1.02, "the range most tools default to", ha="center", fontsize=10,
        color=purple)
ax.annotate("the prompt is\npolitely ignored", xy=(1.7, 0.14), xytext=(1.3, 0.66),
            fontsize=10, color="#444444",
            arrowprops=dict(arrowstyle="-|>", color="#8f8a9c", lw=1.2))
ax.annotate("rigid, oversaturated,\nand overcooked", xy=(17.5, 0.14), xytext=(13.4, 0.36),
            fontsize=10, color="#444444",
            arrowprops=dict(arrowstyle="-|>", color="#8f8a9c", lw=1.2))

ax.set_xlim(1, 20)
ax.set_ylim(0, 1.16)
ax.set_xlabel("guidance scale")
ax.set_ylabel("how well it does")
ax.set_yticks([])
ax.legend(loc="center right", frameon=False)
for side in ("right", "top"):
    ax.spines[side].set_visible(False)
ax.text(1.2, -0.24, "a sketch of the trade-off rather than measured values",
        fontsize=9, color="#666666")

plt.tight_layout()
plt.show()

*Figure: Raising the guidance scale makes the picture follow the prompt more literally and look less natural, so the useful settings sit where the two curves cross.*

Not every tool exposes these controls. Many consumer image services hide the seed, the guidance value, and the step count, so the experiments in this week's lab cannot be run inside them. A hosted open-weight model, opened as a free demo page in a browser, exposes all three without an installation and without a subscription. Check that a tool shows you a seed field before you choose it for this week.

One discipline matters more than any of the individual settings: change one knob at a time. If you change the prompt *and* the seed *and* the guidance, the new picture tells you nothing, because you cannot attribute the difference to anything. That is the only way the tool becomes legible. A locked seed turns a slot machine into an experiment.

## Prompting for images

Image prompts are not language prompts. A language model reads your sentence as instructions; an image model reads it much closer to a bag of weighted concepts, so grammar buys you less and specific nouns buy you more. A working pattern:

```text
[Subject] | [composition] | [style] | [medium] | [lighting] | [mood] | [extra refs]
```

A worked example:

```text
A wooden rowing boat moored at a fjord pier, viewed from a low angle,
black-and-white film photograph, soft early morning light, 50 mm,
nostalgic, in the style of late twentieth-century Scandinavian photography
```

Four habits do most of the work.

- **Be concrete about the subject** before you reach for style. "A car" is vague, and vagueness is answered with the average of everything in the training data. "A rusted 1970s estate car parked in front of a yellow wooden house" is something the model can grip.
- **Use style words deliberately.** Material (oil painting, pen drawing), medium (photograph, render), and period (1970s, Renaissance) each do real work, and they do it more reliably than adjectives like "beautiful" or "high quality".
- **Avoid contradictions.** "Photo-realistic illustration in watercolour" asks the model to choose between three things it cannot do at once, and it will choose for you.
- **Iterate on what is wrong**, not on the whole prompt. If the lighting is off, change only the lighting words. This is the change-one-knob rule applied to the sentence itself.

### Negative prompts

Where a tool supports them, negative prompts are where you shut down the failure modes you keep seeing: blurring, extra fingers, watermarks, deformed faces. Treat the negative prompt as a short list built from your own failures, not as a copied block of a hundred words. A long negative prompt is mostly superstition, and it costs you conditioning capacity that the positive prompt could be using.

### Reference images and control signals

When words run out, show the model what you want. Most tools accept one or more of the following, which are the conditioning channels from chapter [2](how-it-works.ipynb) exposed as interface controls.

- **A style reference**: make it look like this.
- **A structural reference**: match this composition.
- **A pose reference**: put the figure in this position.
- **A depth or edge map**: preserve this geometry, change everything else.

In open-weight tooling these are called ControlNets, small networks trained to inject a spatial signal into the denoiser, and a node-based interface will let you stack several of them at once. Commercial products expose a subset of the same thing under friendlier names such as style reference or character reference.

This is where image generation stops guessing at what you meant and becomes a controllable instrument, and where the honest version of the craft lives. A picture made from your own photograph, your own sketch, and your own depth map is a different artefact from a picture made from a sentence, both aesthetically and ethically. The process memo is where you say which one you made.

## Editing instead of generating

For most professional work you will get further by editing an image than by generating one. The same model runs in all of these modes; only the mask and the starting noise change.

- **Inpainting.** Mask a region and replace it. Swap the object someone is holding, take out a stray sign, repair a mangled hand.
- **Outpainting.** Extend the canvas beyond its original edges, which is how a portrait crop becomes a banner.
- **Variation.** The same subject, slightly moved, for choosing between near-identical takes.
- **Upscaling.** Increase resolution while inventing plausible detail. Note the word *inventing*: an upscaler does not recover information, it fabricates something consistent with what is there.
- **Object removal.** A one-click case of inpainting, and the one your phone already does.

In [ ]:
import numpy as np
from matplotlib.patches import Rectangle

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"
rng = np.random.default_rng(5)

size = 96
ys, xs = np.mgrid[0:size, 0:size]
picture = np.zeros((size, size, 3))
sky = ys / size
picture[..., 0] = 0.36 + 0.5 * sky
picture[..., 1] = 0.42 + 0.44 * sky
picture[..., 2] = 0.62 + 0.32 * sky
sun = (xs - 66) ** 2 + (ys - 70) ** 2 < 11 ** 2
picture[sun] = (0.98, 0.86, 0.55)
for left, width, height, tone in [(6, 16, 40, 0.30), (24, 12, 55, 0.22),
                                  (38, 18, 34, 0.36), (58, 14, 46, 0.26),
                                  (74, 16, 30, 0.33)]:
    picture[0:height, left:left + width] = (tone, tone * 0.9, tone * 1.15)
picture = np.clip(picture, 0, 1)[::-1]

fig, axes = plt.subplots(1, 4, figsize=(10, 3.4))
titles = ["Inpainting", "Outpainting", "Variation", "Upscaling"]
glosses = ["mask a region\nand replace it", "extend past\nthe original edge",
           "the same subject from\na partly noisy copy", "invent detail that\nwas never there"]

axes[0].imshow(picture)
axes[0].add_patch(Rectangle((34, 30), 26, 30, facecolor="white", alpha=0.55,
                            edgecolor=rose, linewidth=2))

pad = 26
wide = np.ones((size + 2 * pad, size + 2 * pad, 3)) * 0.93
wide[pad:pad + size, pad:pad + size] = picture
axes[1].imshow(wide)
axes[1].add_patch(Rectangle((pad, pad), size, size, facecolor="none",
                            edgecolor=rose, linewidth=1.6, linestyle="--"))

noisy = np.clip(picture * 0.65 + rng.normal(0.5, 0.22, picture.shape) * 0.35, 0, 1)
axes[2].imshow(noisy)

blocks = picture.reshape(12, 8, 12, 8, 3).mean(axis=(1, 3))
coarse = np.repeat(np.repeat(blocks, 8, axis=0), 8, axis=1)
combined = coarse.copy()
combined[:, size // 2:] = picture[:, size // 2:]
axes[3].imshow(combined)
axes[3].axvline(size // 2, color=rose, lw=2)
axes[3].text(size * 0.25, 6, "before", ha="center", va="top", fontsize=9,
             color="white")
axes[3].text(size * 0.75, 6, "after", ha="center", va="top", fontsize=9,
             color="white")

for ax, title, gloss in zip(axes, titles, glosses):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title, fontweight="bold", color=purple, pad=8)
    ax.set_xlabel(gloss, color="#444444", labelpad=6)
    for side in ax.spines.values():
        side.set_visible(False)

plt.tight_layout()
plt.show()

*Figure: Four editing modes running on the same picture, and in every one of them the model is asked to fill a gap that you defined rather than to invent a whole image.*

Editing keeps you closer to authorship, because the thing being changed is something you chose or made. It is also the workflow most likely to survive whatever the tools look like in five years, since the underlying operation, mask and regenerate, has been stable since 2022.

## Consistency across a series

A single striking image is easy. Getting four images to show *the same* character, place, or object is the problem that separates a demonstration from a piece of work, and it is where most student projects run into trouble. The reason is structural: the seed sets the noise, the prompt sets the concepts, and nothing in either of them is an identity. Ask twice for "the same woman" and you get two women who match the description.

Three tools address this, in ascending order of effort.

**Character reference images** condition the generation on a picture of the subject, so the model is matching an appearance rather than reconstructing one from words. This is the right tool when you have one good image of the subject and need three or four more of it, and when close enough is good enough. It is the cheapest option, it needs no training, and it drifts under large changes of pose or lighting, because the model is being asked to generalise from a single view.

**Seed locking** holds the noise fixed and changes only the prompt, so the composition, palette, and lighting stay put while the content shifts. This is the right tool when what you need to hold constant is the *frame* rather than the subject: the same street in four seasons, the same room at four times of day. It gives you a visually coherent set almost for free. It will not keep a face the same, because a face is content, not framing.

In [ ]:
import numpy as np
from matplotlib.patches import Ellipse

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"

seeds = [11, 12, 13]
prompts = [("in autumn", ["#f6efe2", "#c88a3d", "#8f5b2a", "#5d4326"]),
           ("at night", ["#1f2545", "#3d3f6b", "#5a2a7a", "#8f8ac4"]),
           ("in snow", ["#f2f5fa", "#b9c8dc", "#7d94b4", "#4a5a72"])]

fig, axes = plt.subplots(len(seeds), len(prompts), figsize=(10, 4.4))

for row, seed in enumerate(seeds):
    rng = np.random.default_rng(seed)
    shapes = [(rng.uniform(0.15, 0.85), rng.uniform(0.15, 0.85),
               rng.uniform(0.2, 0.55), rng.uniform(0.2, 0.55),
               rng.integers(1, 4)) for _ in range(5)]
    for column, (prompt, palette) in enumerate(prompts):
        ax = axes[row, column]
        ax.set_facecolor(palette[0])
        for x, y, width, height, tone in shapes:
            ax.add_patch(Ellipse((x, y), width, height,
                                 facecolor=palette[int(tone)], edgecolor="none",
                                 alpha=0.85))
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xticks([])
        ax.set_yticks([])
        for side in ax.spines.values():
            side.set_visible(True)
            side.set_color("#cfc7dd")
        if row == 0:
            ax.set_title(prompt, color=purple, fontweight="bold", pad=8)
        if column == 0:
            ax.set_ylabel(f"seed {seed}", color=navy, labelpad=8)

fig.text(0.5, 0.015, "the seed fixes the arrangement along a row, and the prompt "
         "changes what fills it", ha="center", fontsize=10, color=navy)
plt.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

*Figure: Nine sketches in which the seed is held along each row and the prompt changes along each column, so the arrangement stays put while the palette moves.*

**LoRA fine-tunes**, where LoRA is short for low-rank adaptation, take a small number of images of your subject, typically in the tens, and train a small set of extra weights that ride on top of a frozen open-weight model. [Fine-tuning](https://en.wikipedia.org/wiki/Fine-tuning_(deep_learning)) of this kind makes the subject a word the model knows. This is the right tool when the same character has to appear across many images over weeks, which is why it is standard practice in comics, animation, and brand work. It is also the only one of the three that genuinely learns your subject. It costs you an open-weight model, a training run, and a rights question, since those training images have to be yours to use.

For a four-image series made in a 90-minute lab, seed locking plus a character reference will get you most of the way. Reach for a fine-tune only when the series outlives the week.

## Where image models still struggle

The failure list shrinks every year, so read this as a snapshot rather than a permanent limit.

- **Hands, feet, jewellery, and complicated logos.** Anything with a strict internal count or structure that the training data mostly shows in passing.
- **Faithful portraits of named individuals.** Technically difficult and, in most commercial tools, deliberately restricted.
- **Multi-step composition.** "A man holds a cat in one hand and points with the other at a sign reading *Open*" asks for several spatial relations at once, and models reliably drop one.
- **Text inside pictures.** Much improved since 2023 and still not solved, especially for anything longer than a few words or in a language other than English.
- **Diagrams and infographics.** A model can produce something with the exact visual grammar of a chart and no relationship at all to any data. This failure is unusually dangerous because the output looks authoritative.
- **Anything requiring a count.** Five chairs is a request the model treats as a suggestion.

The pattern predicts the next failure you meet. These models are excellent at *texture and plausibility* and weak at *discrete structure*. Wherever a correct answer depends on counting, spelling, or a fact rather than on looking right, expect to check it yourself.

:::{note} Dig deeper: generating locally
:class: dropdown
Everything in this chapter can be done in a browser. Running a model on your own machine is what lets you fix the seed with confidence, batch a hundred variations overnight, and keep working when a service changes its terms. It is also the only way to use a fine-tune you trained yourself.

The `diffusers` library is the standard route [@HuggingFaceDiffusers]. The listing below is not executed when this book is built, because it needs a GPU and a model download of several gigabytes. Install the `diffusers`, `transformers`, and `torch` packages, and it runs as it stands.

```python
import torch
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype=torch.float16,
).to("cuda")  # or "mps" on a recent Mac, or "cpu" if you are patient

generator = torch.Generator(device="cuda").manual_seed(42)

img = pipe(
    "a wooden rowing boat at sunrise on a fjord, watercolour",
    num_inference_steps=4,
    guidance_scale=1.0,
    generator=generator,
).images[0]
img.save("boat.png")
```

Two things to notice. The `generator` line is seed locking made explicit: pass the same number and you get the same picture back, which is what makes the grids in this week's lab into an experiment rather than a collection. And `num_inference_steps=4` is only sane because this is a distilled model; on a full model the same value gives you noise with ambitions.
:::

## From stills to moving pictures

A video is a sequence of frames. Generating a sequence of frames that are *consistent over time*, with the same character, the same lighting, and the same physics, is dramatically harder than generating any single frame. In text and images the interface is where most of the interesting decisions live, because the model will usually give you something usable and the craft is in steering it. In video the model itself is the constraint. How long a clip can be, why it drifts, and why the last frame matters all follow from the machinery rather than from a design choice in the tool. Three problems compound.

**Cost.** Five seconds at 30 frames per second is 150 frames. Even in the compressed latent space, that is roughly a hundred times the work of a single image, and the [attention](https://en.wikipedia.org/wiki/Attention_(machine_learning)) that ties the frames together grows faster still. Everything about how video tools are priced, rationed, and queued follows from this one fact.

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"

seconds = np.arange(1, 21)
frames = 30 * seconds
pairs = frames * (frames - 1) / 2

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(seconds, frames, color=purple, lw=2.6, marker="o", markersize=4,
        label="frames to generate")
ax.plot(seconds, pairs, color=rose, lw=2.6, marker="o", markersize=4,
        label="pairs of frames that have to agree")

for value, colour, offset in ((frames[4], purple, 1.6), (pairs[4], rose, 1.6)):
    ax.plot(5, value, "o", color=colour, markersize=9)
    ax.text(5.4, value * offset, f"{int(value):,}".replace(",", " "),
            fontsize=10, color=colour, fontweight="bold")
ax.axvline(5, color="#b9b4c4", lw=1.2, ls="--")
ax.text(5.0, 40, "a five-second clip", ha="center", va="top", fontsize=10,
        color="#444444")

ax.set_yscale("log")
ax.set_ylim(20, 3e5)
ax.set_xlim(1, 20)
ax.set_xticks([1, 5, 10, 15, 20])
ax.set_xlabel("length of the clip in seconds, at 30 frames a second")
ax.set_ylabel("count (logarithmic scale)")
ax.legend(loc="lower right", frameon=False)
ax.grid(color="#dcd7e4", lw=0.8, ls=":")
ax.set_axisbelow(True)
for side in ("right", "top"):
    ax.spines[side].set_visible(False)

plt.tight_layout()
plt.show()

*Figure: The frames in a clip grow with its length, and the pairs of frames that have to agree grow with the square of it, which is where the cost of video sits.*

**Consistency.** Each frame has to agree with the frames around it. A jacket cannot change colour halfway through, a face cannot subtly become a different face, and a shadow cannot swap sides. That requires the model to carry information across the sequence rather than to solve each frame on its own, which is why coherence degrades as clips get longer.

**Physics.** Cloth has to fall, water has to flow, a hand has to close around a cup and stay closed. An image model can fake plausible physics in a single frame, because a still only has to look right once. A video model has to fake a *trajectory*, and a trajectory has many more ways of being wrong. People are unusually sensitive to the motion of bodies and faces, so a trajectory that is slightly wrong reads as wrong long before anyone can say what changed. A still image is judged by what it shows; a clip is judged by whether its motion is believable.

Three strategies address the time axis, and the failures you meet are strategy-specific, so the names matter.

- **Frame-by-frame with attention across frames.** Frames are generated in sequence, and each one attends to the others so that it can see what has already been drawn. This is comparatively cheap, and it drifts, because small errors accumulate in the direction of travel.
- **Space-time diffusion.** The whole clip is treated as one block of numbers with two spatial dimensions and one temporal one, and the entire block is denoised at once. Conceptually clean, globally consistent, and expensive enough that clip length is capped hard.
- **Two-stage keyframe and interpolation.** A first model generates a small number of keyframes, and a second lighter model fills in the frames between them. This buys length cheaply, and it produces the characteristic failure where the keyframes are excellent and the motion between them slides.

In [ ]:
from matplotlib.patches import FancyBboxPatch, Rectangle

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"

fig, ax = plt.subplots(figsize=(10, 5))
rows = [
    (4.5, "Frame by frame, with attention across frames",
     "cheap, and it drifts in the direction of travel"),
    (2.6, "Space-time diffusion",
     "consistent throughout, and the clip length is capped hard"),
    (0.7, "Keyframes, then interpolation",
     "long and cheap, and the motion between the keyframes slides"),
]

for index, (base, heading, note) in enumerate(rows):
    ax.text(0.3, base + 1.05, heading, fontsize=11, fontweight="bold", color=navy)
    for i in range(5):
        x = 0.3 + i * 1.1
        if index == 2:
            keyframe = i in (0, 3)
            face = "#fff0f3" if keyframe else "white"
            edge = rose if keyframe else "#b9b4c4"
            style = "-" if keyframe else "--"
        else:
            face, edge, style = "#f4f1fa", purple, "-"
        ax.add_patch(Rectangle((x, base), 0.85, 0.8, facecolor=face,
                               edgecolor=edge, linewidth=1.6, linestyle=style))
        if index == 0 and i < 4:
            ax.annotate("", xy=(x + 1.1, base + 0.4), xytext=(x + 0.85, base + 0.4),
                        arrowprops=dict(arrowstyle="-|>", color=navy, lw=1.4))
    if index == 1:
        ax.add_patch(FancyBboxPatch((0.15, base - 0.15), 5.55, 1.1,
                                    boxstyle="round,pad=0.02,rounding_size=0.12",
                                    facecolor="none", edgecolor=rose,
                                    linewidth=1.8, linestyle="--"))
    ax.text(5.9, base + 0.4, note, va="center", fontsize=10, color=navy)

ax.text(0.3, 6.2, "Three ways to handle the time axis, and what each one gets wrong",
        fontsize=11, fontweight="bold", color=purple)

ax.set_xlim(0, 12)
ax.set_ylim(0.2, 6.7)
ax.axis("off")
plt.tight_layout()
plt.show()

*Figure: Each of the three strategies for the time axis buys something and fails in its own way, which is why the failures are worth recognising by name.*

The strongest public systems combine all three, which is why their outputs fail in mixed ways rather than in one recognisable way.

## What video models can do

The list below is a snapshot and will date faster than most of this book. Read it as the shape of the capability rather than as a specification.

Mainstream text-to-video and image-to-video services reliably produce the following.

- **Short clips.** Roughly five to fifteen seconds at high definition, sometimes longer at reduced quality [2026?].
- **Photorealistic or stylised output.** Selected by prompt, in the same way as an image model.
- **Camera motion under instruction.** Orbit, dolly, push, pull, and hand-held.
- **Image-to-video.** A still you already have is animated.
- **Keyframe-to-keyframe.** You supply a first and a last frame, and the model invents the motion between them.
- **Lip-sync.** A static face is driven by an audio file.
- **Style transfer across a clip.** Footage you shot yourself is restyled.

They struggle with five things.

- **Length.** Coherence falls away somewhere past ten or twenty seconds in most systems.
- **Specific people and protected characters.** Most commercial services refuse these outright rather than fail at them.
- **Crowds and interaction.** Two people holding a conversation is at the edge of what is reliable.
- **Hands, small objects, and printed text.** These are the image failures above with a time axis added to them.
- **Counting and continuity.** The number of people in a shot can change while the shot is running.

The pattern is the same one as for images. These models are excellent at texture and plausibility and weak at discrete structure, and adding time gives the weakness more room to show. If your shot depends on something being *the same* from second one to second eight, plan to check it.

Two consequences matter for how you work. The honest unit of video AI is the shot, not the scene and certainly not the film, because almost every system generates a single continuous take. And image-to-video is usually the better route than text-to-video, for the reason that a reference image beat a longer prompt above. A still you have already judged removes an entire category of uncertainty before the expensive part begins.

### Cost and access

Video is the most expensive medium per second of output, and this shapes the practical experience more than anything else in this chapter. A few seconds of video costs, in round terms, on the order of a hundred image generations. Consumer services therefore meter video in credits rather than in requests, and a single clip consumes a noticeable fraction of a monthly allowance. On a free tier, assume a handful of attempts rather than a session of experimentation. Prices and allowances change every few months, so check the current terms rather than trusting any figure printed in a book, including this one.

Two consequences for your practice. Plan before you generate, because in this medium iteration is a budget rather than a habit, and the storyboard is what stops you spending five clips discovering what one sketch would have told you. And do the work in a hosted tool unless you have a workstation with a serious accelerator. Local video generation is possible with open-weight models, and on a laptop with integrated graphics it is a way of watching a progress bar rather than a way of making a film.

## Prompting for video

Video prompts are richer than image prompts because they have to describe change as well as appearance. A working template:

```text
[Subject + key features],
[scene / setting],
[camera angle and movement],
[lighting and time of day],
[style],
[motion within the scene]
```

A worked example:

```text
A young woman walking quickly along the riverbank in Oslo,
autumn leaves on the ground, river to the left,
medium-wide shot from a hand-held camera following her from behind,
overcast late-afternoon light,
documentary style,
she pulls a beanie out of her pocket and puts it on as she walks
```

Four concepts do most of the work.

**Camera moves.** The vocabulary is borrowed straight from film, and the models were trained on footage that came with those words attached, so using them properly pays. *Dolly in*, *truck left*, *crane up*, *orbit*, *push in*, *static*. A named move is understood far more reliably than a description of the same move in ordinary language.

**Motion, spelled out in time.** Do not assume that "looking sad" will produce someone beginning to cry. State what moves, in what order, and roughly how fast. This is the single largest difference from image prompting, and the most common reason a clip comes back with a beautiful subject standing perfectly still.

**Cuts, which you do not get.** Most systems generate one continuous shot, so a request for a cut inside a prompt is either ignored or answered with a bewildering camera lurch. Multi-shot sequences are assembled afterwards in an editor, which means the edit is yours and always was.

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"
rng = np.random.default_rng(12)

count, height, width = 60, 36, 48
cut = 34
ys, xs = np.mgrid[0:height, 0:width]
clip = np.zeros((count, height, width))

for index in range(count):
    if index < cut:
        base = 0.12 + 0.16 * ys / height
        centre = 10 + 0.75 * index
        blob = 0.85 * np.exp(-(((xs - centre) / 5.0) ** 2 + ((ys - 20) / 5.0) ** 2))
    else:
        base = 0.92 - 0.22 * ys / height
        centre = 38 - 0.55 * (index - cut)
        blob = -0.75 * np.exp(-(((xs - centre) / 4.0) ** 2 + ((ys - 12) / 4.0) ** 2))
    clip[index] = np.clip(base + blob + rng.normal(0, 0.012, (height, width)), 0, 1)

difference = np.abs(np.diff(clip, axis=0)).mean(axis=(1, 2))

fig = plt.figure(figsize=(10, 4.6), layout="constrained")
grid = fig.add_gridspec(2, 8, height_ratios=[1, 1.6])
samples = [0, 10, 20, 30, 34, 40, 50, 59]
for column, index in enumerate(samples):
    ax = fig.add_subplot(grid[0, column])
    ax.imshow(clip[index], cmap="bone", vmin=0, vmax=1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"frame {index}",
                 color=rose if index >= cut else navy)
    for side in ax.spines.values():
        side.set_visible(True)
        side.set_color(rose if index >= cut else "#cfc7dd")

curve = fig.add_subplot(grid[1, :])
curve.plot(np.arange(1, count), difference, color=purple, lw=2.2)
curve.axvline(cut, color=rose, lw=1.6, ls="--")
curve.annotate("the cut", xy=(cut, difference[cut - 1]), xytext=(cut + 6, difference.max() * 0.78),
               fontsize=11, color=rose,
               arrowprops=dict(arrowstyle="-|>", color=rose, lw=1.4))
curve.text(4, difference.max() * 0.30, "steady movement inside one shot",
           fontsize=10, color="#444444")
curve.set_xlabel("frame")
curve.set_ylabel("average change\nfrom the frame before")
curve.set_xlim(1, count - 1)
for side in ("right", "top"):
    curve.spines[side].set_visible(False)

plt.show()

*Figure: A synthetic clip with a cut at frame 34, and the average change from one frame to the next below it, where the cut is the one place the curve leaves the floor.*

**Aspect ratio as a compositional decision.** 16:9 for landscape work, 9:16 for the vertical feeds. A composition built for one shape does not survive being cropped into the other, and framing a walking figure for a vertical frame is a different job from framing them for a wide one.

The change-one-knob discipline applies here with more force, because each generation costs more and you get fewer of them. If you change the motion words and the camera words at once, you have spent two generations to learn nothing.

## Where video fits in a production

A realistic pipeline has five steps, and only two of them are generative.

1. **[Storyboard](https://en.wikipedia.org/wiki/Storyboard)** with an image model until you have a frame you are happy with for every shot. This is where seed locking and reference images earn their keep, because shots that share a look at storyboard stage are much likelier to cut together later.
2. **Animate** each approved frame with image-to-video, using end-frame conditioning where the shot has a clear destination.
3. **Stitch** the shots in a video editor. A free, professional-grade editor is enough for everything in this course.
4. **Add audio**: voice, music, [Foley](https://en.wikipedia.org/wiki/Foley_(filmmaking)), and ambience, which chapter [6](sound.ipynb) teaches you to make.
5. **Grade and finish** in the same editor, which is also where you add the provenance card.

In [ ]:
from matplotlib.patches import FancyBboxPatch

import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.figsize": (10, 4),
    "figure.dpi": 100,
    "savefig.dpi": 150,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

navy, purple, rose = "#1f2545", "#5a2a7a", "#e8556d"

steps = [("1 Storyboard", "an image model", True),
         ("2 Animate", "image to video", True),
         ("3 Stitch", "a video editor", False),
         ("4 Add audio", "voice, music, Foley", False),
         ("5 Grade\nand finish", "the same editor", False)]

fig, ax = plt.subplots(figsize=(10, 3.6))
width, gap = 2.1, 0.35
for index, (name, gloss, generative) in enumerate(steps):
    x = 0.3 + index * (width + gap)
    ax.add_patch(FancyBboxPatch((x, 1.5), width, 1.35,
                                boxstyle="round,pad=0.02,rounding_size=0.14",
                                facecolor="#f4f1fa" if generative else "#fbfaff",
                                edgecolor=purple if generative else "#cfc7dd",
                                linewidth=2.0 if generative else 1.5))
    ax.text(x + width / 2, 2.36, name, ha="center", va="center", fontsize=11,
            fontweight="bold", color=navy, linespacing=1.3)
    ax.text(x + width / 2, 1.85, gloss, ha="center", va="center", fontsize=10,
            color="#444444")
    if index < len(steps) - 1:
        ax.annotate("", xy=(x + width + gap - 0.05, 2.175), xytext=(x + width + 0.05, 2.175),
                    arrowprops=dict(arrowstyle="-|>", color=navy, lw=1.6))

ax.plot([0.3, 4.75], [1.15, 1.15], color=purple, lw=2.4)
ax.text(2.5, 0.85, "generated", ha="center", fontsize=11, fontweight="bold",
        color=purple)
ax.plot([5.2, 12.4], [1.15, 1.15], color=rose, lw=2.4)
ax.text(8.8, 0.85, "edited, and where most of the perceived quality comes from",
        ha="center", fontsize=11, fontweight="bold", color=rose)

ax.set_xlim(0, 12.7)
ax.set_ylim(0.5, 3.1)
ax.axis("off")
plt.tight_layout()
plt.show()

*Figure: Only the first two steps of a realistic video pipeline are generated, and the rest is editorial work that decides most of what an audience notices.*

The parts most transformed by AI are storyboarding and animation. The editorial work, which is pacing, sound design, and colour, remains very human, and it is also where most of the perceived quality of a finished piece comes from. A weak edit ruins good shots, and a strong edit rescues mediocre ones.

```{tip}
Plan the last frame at the same time as the first. A model given both ends has a defined destination and interpolates towards it, whereas a model given only a start has to invent where the shot is going and will often invent something you did not want. End-frame conditioning is frequently what separates a magic clip from a useful one.
```

## Talking heads and consent

One genre of video AI deserves separate treatment: the talking head. Upload a photograph or a short clip of a person, or pick one from a stock library, and drive it with synthetic speech. You get a video of that person saying anything you type, with [lip-sync](https://en.wikipedia.org/wiki/Lip_sync), and a multilingual version of the same presenter is a checkbox.

The legitimate uses are real and dull, which is a good sign for a technology. Corporate training, product explainers, translated versions of a recorded lecture, and localised marketing all involve saying scripted words to camera. That is a task that is expensive to film and cheap to fake, and one where nobody is deceived about what they are watching.

The same tool is the [deepfake](https://en.wikipedia.org/wiki/Deepfake) tool, close to ideal for personal and political manipulation, and the distance between the two uses is not technical but a matter of consent and labelling. Chapter [6](sound.ipynb) makes this argument for the voice, and every word of it applies to the face, with the aggravation that a face is harder to deny than a voice and easier to circulate. Chapter [3](co-creation-and-ethics.ipynb) sets out where the law has reached, including the transparency obligations in the [EU AI Act](https://en.wikipedia.org/wiki/Artificial_Intelligence_Act) for material of this kind [@EUAIAct; @EUAIActSummary].

The working rule for this course is short. If a real person appears, you have their explicit, informed, written consent for this specific use, and the output is labelled where a viewer will see it. There is no version of the exercise that is worth breaking that rule for.

## World models

Something is converging that was not obviously the same problem five years ago. A video model predicts what the next frames look like. A [game engine](https://en.wikipedia.org/wiki/Game_engine) computes what the next frame looks like given what the player just did. A model that predicts the next frame *conditioned on an action* is doing both at once, and what you have then is not a clip but a place you can move around in.

The clearest public demonstration is Genie, which learned from unlabelled internet video of people playing games and produced an interactive environment that responds to a latent action input, without ever being shown which button caused which change [@Bruce2024]. The controls were not annotated in the training data; the model inferred that some component of the change between frames was caused by an agent, and exposed it as something you can drive. Systems along these lines are called [world models](https://en.wikipedia.org/wiki/World_model): generative models whose output is a navigable state rather than a finished artefact.

For games, this points at content that is generated as it is played rather than authored in advance, and at prototyping a playable feel before any assets exist. Both are useful and neither is close to replacing a designed level, because the thing a world model is worst at is precisely the thing a game most needs, which is a rule that holds. Chapter [7](spatial.ipynb) takes up spatial and three-dimensional generation directly, and the tension there is the same one. A scene you can walk through has to stay put while you walk through it, and a model that predicts plausible next frames has no obligation to keep anything. A world model that is impressive to watch and one that is consistent enough to build on are different achievements, and the public demonstrations are much further along on the first than the second.

:::{note} Dig deeper: what the systems are called
:class: dropdown
The body of this chapter describes categories rather than products, because the products change faster than the book can be revised and the categories have been stable since 2024. For the record, and because you will meet the names immediately when you start searching, the systems that defined this period include Sora, Veo, Runway, Kling, Pika, and Luma for general text-to-video and image-to-video [2026?]. HeyGen, Synthesia, and D-ID do talking heads. DaVinci Resolve is the free professional editor referred to in the workflow above; Premiere, Final Cut, and CapCut do the same job.

Two of these are worth reading rather than only using. The Sora system card is the developer's own account of what the model refuses, what it gets wrong, and what it was trained to avoid, and it is unusually candid about limits [@OpenAISora]. The Runway research blog tracks the model-side developments in a form aimed at people who make things [@RunwayResearch].
:::

:::{note} Dig deeper: will AI video replace film crews?
:class: dropdown
The claim is that video generation is on the same curve images were on in 2022. Within a few years, it goes, a director with a laptop will make a feature, and the crew, the camera department, and the extras will go the way of the darkroom technician.

Look at what has shipped, because that is a shorter and more informative list than the projections. Generated video is in commercial use for advertising inserts, [B-roll](https://en.wikipedia.org/wiki/B-roll) in documentary and news-adjacent work, music videos, title sequences, and [previsualisation](https://en.wikipedia.org/wiki/Previsualization), the rough version of a sequence made before the real shoot in order to plan it. In each case the shot is short, non-speaking, and not required to match a specific other shot. There is no known sustained narrative feature film made this way, and not for lack of trying or funding [2026?]. The system card published alongside one of the leading text-to-video models lists physical implausibility, difficulty with cause and effect over time, and confusion in scenes with several interacting subjects as that model's own limits [@OpenAISora]. The Stanford AI Index tracks how fast the underlying capability is moving, year by year, and is a better source than any single launch [@StanfordAIIndex].

Then ask how the impressive examples were produced. Almost everything circulating is a demo reel, and a reel is cut from many attempts, the ratio is almost never reported, and the shots that are kept are exactly the ones where the model happened to hold together. That makes the reel evidence for "this system can sometimes produce a good shot" and not for "this system can produce the shot you need". Production asks for a *specific* shot, matching an established look, on a schedule, and asks for it again tomorrow when the client wants the jacket changed. A model with a high ceiling and a low floor is excellent for the first question and unreliable for the second, and the gap between them is where the crew still lives. When you evaluate a claim in this area, look for the denominator: how many generations were run, how many were kept, and who chose.

So the honest version of the claim is narrower and still substantial. Storyboarding and animation, the front end, have been transformed, and AI has made them possible for people who previously could not afford to try at all. Pacing, sound, and colour have not, because those tasks are about judgement against an intention rather than plausibility. And whether a crew is employed is a decision made by people with budgets, as chapter [3](co-creation-and-ethics.ipynb) argues. A tool that removes the cheapest tier of work first will be felt first by the people with the least security in the industry, whether or not it is good enough to replace them. "Can it?" and "will they?" are different questions, and the second is not answered by watching a reel.
:::

:::{admonition} Research spotlight
:class: spotlight
The [Musical Gestures Toolbox for Python](https://github.com/fourMs/MGT-python) is an open-source library from the fourMs Lab at UiO for analysing video of people moving, most often musicians [@MGTpython]. Give it a recording and it computes a motiongram: a single still image in which one axis is time and the other is space, built by collapsing the frame-to-frame differences of the video into a strip. A short clip of a violinist becomes a picture in which the bow strokes are visible as a rhythm of bright bands. A videogram does the same with the original pixels rather than the motion, so a ten-minute performance arrives as one image you can take in at a glance.

Reading these pictures is a skill, and it comes quickly. Steady playing at a steady tempo appears as an even stripe pattern, and the spacing of the stripes is the beat, so a tempo change is visible as the point where the stripes crowd together or spread apart. A held note is a long smear with nothing crossing it. Researchers in the lab use this to find the interesting thirty seconds inside two hours of rehearsal footage, and to compare two performances of the same piece by stacking their strips one above the other. Put a generated clip through the same view and drift, which this chapter has described in the abstract, becomes a shape you can point at. A horizon that ought to run straight along the strip wanders, a colour that ought to hold slides across the width, and a model that interpolated between keyframes leaves seams at regular intervals.

These are pictures made by a computer, and they have nothing to do with generation. Every pixel is a measurement. If the player stops moving, the motiongram goes flat, and a videogram cannot contain a gesture the camera did not see, because every column of it was computed from a frame that exists. A generated image or clip has no such obligation. It answers "what would a plausible picture of this look like", and it succeeds when it looks right, whether or not anything corresponding to it ever happened. Both are legitimate kinds of picture, and confusing them is how a generated illustration ends up captioned as evidence, and why a synthetic clip in a news bulletin is a category error before it is an ethical one. Chapter [10](body.ipynb) returns to motion capture and the body.

The toolbox installs with `pip` and runs on any video file you have. If visualising motion or moving image is where your project is heading, put one of your own clips through it, bring the strip to the lab, and we will point you at the people at fourMs who work on it.
:::

## This week's lab: Explore, Reflect, Create

This lab has three movements: find out what each knob does, for a still and then for a shot; argue about what the model leaves out; and make something you would put your name on. Video generations are the scarce resource, so the lab budgets five of them for the whole session: one in Explore and up to four in Create. Decide your five before you spend any of them.

### Explore (about 30 min)

**A controlled experiment.** Work in a tool that shows you the seed, the guidance value, and the step count, since a service that hides its settings cannot answer any of the questions below. Write one base prompt with a clear subject, composition, and style, and keep it fixed for the whole exercise.

1. Generate three images with the same prompt and the same seed at three different guidance values, for example 3, 7, and 12.
2. Generate three with the same prompt and three different seeds, changing nothing else.
3. Generate three in which the prompt is unchanged except for the lighting words.
4. Generate three with the same prompt and the same seed at three different aspect ratios.

Lay each set out as a labelled grid, so you leave the lab with four grids rather than twelve loose files. Then answer one question in a sentence: which knob mattered most *for your subject*, and why that subject rather than another.

**One shot from the best cell.** Take the single image you like best and run image-to-video on it with a short motion prompt. Name a camera move and state the motion in time, for example *the subject turns their head slowly to the right while the camera pushes in*. Watch it twice. Write down in one sentence where coherence held and where it broke, and be specific about *when* in the clip it broke, because the moment of failure tells you far more than the fact of it.

**One more generation, for the next block.** Generate a single image of "a typical Norwegian street", and find a photograph of an actual Norwegian street to put beside it, ideally one you took yourself. Bring both to the Reflect round.

**Optional.** Run a photograph you have the rights to through image-to-image at three denoise strengths, around 0.3, 0.5, and 0.8, and mark where on the axis from fidelity to freedom you would want to sit for a real job. Compare your best and worst grid in the [Seed and guidance grid viewer](https://fourms.github.io/Creative-AI/apps/grid-viewer/), and step through your shot in the [Frame-consistency inspector](https://fourms.github.io/Creative-AI/apps/frame-consistency/).

### Reflect (about 15 min)

Work in pairs, then in plenary. This is a discussion, not a writing block.

1. Put your "typical Norwegian street" generation beside the photograph you brought. Ask your partner what has been averaged away: which decade, which weather, which parts of the country, which people, which mess. Be specific, because "it looks generic" is the answer that stops the conversation rather than starting it.
2. Watch each other's shot in silence, twice, and say three things out loud, in this order: one thing the model clearly did well, one thing that gives it away, and one thing that would be the next step.
3. Close the round by stating aloud what you are going to make in the Create block: the subject and the through-line. For a series, say which of the three consistency tools you will use; for a sequence, say what the cut between shot one and shot two is doing.

### Create (about 45 min)

Assemble one finished piece from what you generated above. This is a portfolio artefact, not an exercise. Choose one form:

- a four-image series with a shared subject and a clear conceptual through-line, for example the same Oslo street in four seasons, or a photograph and redraw diptych that puts a real image next to its image-to-image variant;
- a three-shot sequence of about twenty seconds on one small idea, for example a tourist arriving in Oslo at sunrise. Storyboard three frames with the image model, and animate each with image-to-video from its frame rather than from a sentence. Cut them together in a video editor, paying attention to the cut points, because that is the part of the pipeline the model cannot do for you. If your allowance runs out, animate one shot and hold the other two as stills with a slow push or pan added in the editor; one generated shot cut against two moving stills is a complete answer to this brief.

Whichever you choose, record the provenance. For the series, write a 100-word honest caption recording the tool, the model, the seed where you can see it, the prompt, the variations you tried, and every human edit. For the sequence, add a provenance card as a final frame, white text on black, with the tools, the models, the year, and what is generated. For material of this kind the card is moving towards being required under the EU AI Act [@EUAIAct; @EUAIActSummary]. The caption or card is part of the artefact, not metadata about it, and a piece that needs it hidden is not finished.

**A2 is due this week.** *AI-assisted text in your discipline* was set in week 4 and is handed in now, so keep an hour clear for it; the piece you make here is separate lab work and does not replace it.

**Optional, for the commons.** If one of your photographs would help explain something, for example a street in a place Wikipedia has no picture of, upload it to [Wikimedia Commons](https://commons.wikimedia.org/). Commons accepts AI-generated images only when they have a realistic educational use, such as showing what a diffusion model does with a given prompt, and they must be labelled as generated. Your own photograph is usually the more useful contribution.

At home, write this week's entry in your practice log using the [practice log template](templates/practice-log.md).

## A critical look: does an image model copy its training images?

**The claim.** A diffusion model is a collage machine. It has stored the pictures it was trained on and reassembles fragments of them on request, so every output is a copy of somebody's work and the only question is how well hidden the seams are.

**The evidence.** There is real evidence for a narrow version of this. In 2023 two research groups showed that training images can be pulled back out of a diffusion model. One team recovered around a hundred near-duplicates of specific training images from the open image model alone. It did so by generating on the order of a hundred million pictures from hundreds of thousands of prompts aimed at candidates it had already identified [@Carlini2023]. Another found copied content in a small percentage of outputs from a widely used open model and traced it to the same cause [@Somepalli2023]. Both point at the same mechanism. The images that come back are overwhelmingly ones that appeared many times in the training set, whether as a stock photo reused across thousands of pages or a famous painting reproduced everywhere. Duplication in the data is what turns a general pattern into a memorised instance.

**The method.** Look at how those findings were produced, because the two studies were built to answer different questions. The extraction team [@Carlini2023] started from candidate images already known to be duplicated in the training set, wrote prompts aimed straight at them, generated many samples per prompt, and ran a nearest-neighbour search against the training data to find matches. That is an adversarial extraction attack, and it is the right design for its question, which is *can a copy be forced out at all*. It is the wrong design for the question people quote it for, which is *how often does this happen in ordinary use*, because a procedure built to maximise a rate cannot also measure it. The replication study [@Somepalli2023] asked that second question instead, sampling generations without aiming at known duplicates and then retrieving the nearest training images to estimate how often a near match turns up unprompted. Read together they say that copies can be extracted deliberately and that they surface rarely otherwise, and neither figure substitutes for the other.

**The limits.** So memorisation is real, rare, and concentrated on images the training set contained many times over. For the overwhelming majority of outputs, "copy" is the wrong word: a model of a few gigabytes cannot be storing the billions of images it saw, and what it holds is a set of statistical regularities, not an archive. The rare exact copy is the exception that shows the rule, since a system built as what chapter [3](co-creation-and-ethics.ipynb) calls an anarchive of relations will occasionally return a record anyway. That is why the duty to record provenance falls on the maker rather than on the model. But this does not settle the ethics. The cases filed from 2023 onwards, among them Andersen v. Stability AI [@AndersenStability] and Getty Images v. Stability AI [@GettyStability], mostly turn on whether *training* on scraped images was lawful, not on whether outputs are copies. A model that never reproduces a single training image still learned everything it knows from work that was taken without asking. Chapter 3 sets out the argument and where the law has reached. The technical answer to "is it a collage" is no. The answer to "is it fair" is not a technical question.

```{admonition} Chapter summary
:class: tip
A diffusion model is trained by adding noise to real images and learning to remove it one step at a time, and it generates by running that reversal from pure noise under a conditioning signal, usually a text prompt. Latent diffusion moves the process into a compressed space, which is why these models run on ordinary hardware and why open-weight image models exist. The same trained network gives you image-to-image, inpainting, and outpainting by changing only where it starts and what it is allowed to touch. The interface exposes a small, transferable vocabulary of prompt, negative prompt, aspect ratio, guidance, steps, seed, sampler, and reference image, and the one discipline that makes any of it legible is to change a single knob at a time. Reference images, control signals, and editing move you from prompting towards authorship; character references, seed locking, and LoRA fine-tunes hold a series together, in ascending order of effort. The models stay weak wherever an answer depends on discrete structure rather than on looking plausible. Video is the same machinery with a time axis, along which cost, consistency, and physics compound, and the three strategies for that axis, attention across frames, space-time diffusion, and keyframes with interpolation, each fail in their own way. The reliable unit is a single shot, and named camera moves and motion stated in time are what a video prompt adds. In a real pipeline the model storyboards and animates while stitching, sound, and grading stay with you, and end-frame conditioning is the most reliable way to make a shot useful. Talking heads are a consent problem before they are a video problem, and world models point at where this is going, since a next-frame predictor conditioned on an action is a playable place rather than a clip. And the models do not store their training images, which settles the collage claim without settling anything about consent.
```

```{admonition} Questions
:class: question
1. Explain forward and reverse diffusion to someone who has never used an image tool, and say why noise has to be part of the story and what latent diffusion changes.
2. State the change-one-knob rule and say what goes wrong without it. Give an example from your own grids where breaking it would have cost you the answer, and say why the rule bites harder for video.
3. Why do two generations of "the same character" drift apart, and which of the three consistency tools would you reach for in a project that runs for a semester?
4. Give the three reasons video generation is harder than image generation, name the three strategies models use for the time axis, and say which failure you saw in your own shot this week.
5. Take a shot you want and write it as a video prompt with a named camera move and a motion stated in time. Then say why planning the last frame improves the shot, in terms of what the model is being asked to do rather than the interface.
```

:::{seealso} Further reading
- **Rombach et al., *High-Resolution Image Synthesis with Latent Diffusion Models* (2022)**: the founding paper of the open-weight image models, and readable if you skip the equations [@Rombach2022LatentDiffusion]. [arxiv.org/abs/2112.10752](https://arxiv.org/abs/2112.10752)
- **Carlini et al., *Extracting Training Data from Diffusion Models* (2023)**: the memorisation study behind the critical look above; read the method section, not the headline [@Carlini2023]. [arxiv.org/abs/2301.13188](https://arxiv.org/abs/2301.13188)
- **Gatys, Ecker, and Bethge, *A Neural Algorithm of Artistic Style* (2015)**: the paper that opened neural style transfer, and the most useful pre-diffusion ancestor to know about [@Gatys2015]. [arxiv.org/abs/1508.06576](https://arxiv.org/abs/1508.06576)
- **[Lev Manovich](https://en.wikipedia.org/wiki/Lev_Manovich), *AI Aesthetics* (2018)**: short essays on what AI image-making *looks like*, written before the current generation and sharper for it [@Manovich2018]. [manovich.net](http://manovich.net/index.php/projects/ai-aesthetics)
- **Aaron Hertzmann, *Can Computers Create Art?* (2018)**: worth re-reading this week with your own generations in hand [@Hertzmann2018]. [doi.org/10.3390/arts7020018](https://doi.org/10.3390/arts7020018)
- **OpenAI, *Sora system card* (2024)**: the developer's own account of what a video model can and cannot do [@OpenAISora]. [openai.com/sora](https://openai.com/sora)
- **Bruce et al., *Genie: Generative Interactive Environments* (2024)**: the world-model paper; read the description of how latent actions were learned without labels [@Bruce2024]. [arxiv.org/abs/2402.15391](https://arxiv.org/abs/2402.15391)
- **Regulation (EU) 2024/1689, the AI Act, and its official summary**: the transparency and labelling obligations that apply to synthetic images and video of people [@EUAIAct; @EUAIActSummary]. [eur-lex.europa.eu](https://eur-lex.europa.eu/eli/reg/2024/1689/oj)
- **Coalition for Content Provenance and Authenticity, *C2PA specification***: the technical standard behind the provenance metadata your caption or card is a human-readable version of [@C2PA]. [c2pa.org/specifications](https://c2pa.org/specifications/)
- **Hugging Face, *Diffusers* documentation**: the fastest route from this chapter to running and fine-tuning a model yourself [@HuggingFaceDiffusers]. [huggingface.co/docs/diffusers](https://huggingface.co/docs/diffusers/index)
:::

:::{tip} Explore interactively
- [Forward diffusion explorer](https://fourms.github.io/Creative-AI/apps/diffusion-explorer/): drag a slider to add noise to an image and watch it dissolve, then run the same slider backwards.
- [Seed and guidance grid viewer](https://fourms.github.io/Creative-AI/apps/grid-viewer/): step through a grid of generations that vary one knob at a time, with the settings shown beside each cell.
- [Frame-consistency inspector](https://fourms.github.io/Creative-AI/apps/frame-consistency/): step through a generated clip frame by frame with a difference view beside it, so you can see when and where the drift begins.
:::